In [1]:
import pandas as pd
import numpy as np
from pgmpy.models import BayesianModel
from pgmpy.models import BayesianNetwork
from pgmpy.inference import VariableElimination, ApproxInference, BeliefPropagation
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.estimators import BayesianEstimator
from pgmpy.estimators import HillClimbSearch
from pgmpy.estimators import BDeuScore, K2Score, BicScore
from pgmpy.metrics import structure_score
from pgmpy.utils import get_example_model
from pgmpy.estimators import ScoreCache
from pgmpy.inference.CausalInference import CausalInference
import networkx as nx
import bnlearn as bn
import itertools
import math
import networkx as nx
import matplotlib.pyplot as plt

In [2]:
from same_decision_probability_calculation import *
from minimum_information_loss_partition import *
from utils import *

from monte_carlo_sdp import *

In [3]:
from pgmpy.utils import get_example_model

# Loading Models

In [4]:
import pandas as pd
from ucimlrepo import fetch_ucirepo
from pgmpy.models import NaiveBayes
from pgmpy.estimators import MaximumLikelihoodEstimator

# ── VOTING ──────────────────────────────────────────────────────────────────
voting = fetch_ucirepo(id=105)
df_voting = pd.concat([voting.data.features, voting.data.targets], axis=1)
df_voting.columns = [c.strip() for c in df_voting.columns]

# Replace '?' missing values — Naive Bayes needs complete data
df_voting = df_voting.replace('?', pd.NA).dropna()

# All values must be strings/categories for pgmpy
df_voting = df_voting.astype(str)

target_voting = 'Class'   # 'democrat' / 'republican'

voting_model = NaiveBayes()
voting_model.fit(df_voting, target_voting,
                 estimator=MaximumLikelihoodEstimator)

# ── CHESS ────────────────────────────────────────────────────────────────────
chess = fetch_ucirepo(id=22)
df_chess = pd.concat([chess.data.features, chess.data.targets], axis=1)
df_chess = df_chess.astype(str)

target_chess = 'skach' 

chess_model = NaiveBayes()
chess_model.fit(df_chess, target_chess,
                estimator=MaximumLikelihoodEstimator)

In [5]:
# cast models to pgmpy BayesianNetwork for compatibility with our code
voting_model = BayesianNetwork(voting_model.edges())
chess_model = BayesianNetwork(chess_model.edges())

# fit
voting_model.fit(df_voting, estimator=MaximumLikelihoodEstimator)
chess_model.fit(df_chess, estimator=MaximumLikelihoodEstimator)

In [6]:
df_voting.head()

,handicapped-infants,water-project-cost-sharing,adoption-of-the-budget-resolution,physician-fee-freeze,el-salvador-aid,religious-groups-in-schools,anti-satellite-test-ban,aid-to-nicaraguan-contras,mx-missile,immigration,synfuels-corporation-cutback,education-spending,superfund-right-to-sue,crime,duty-free-exports,export-administration-act-south-africa,Class
5,n,y,y,n,y,y,n,n,n,n,n,n,y,y,y,y,democrat
8,n,y,n,y,y,y,n,n,n,n,n,y,y,y,n,y,republican
19,y,y,y,n,n,n,y,y,y,n,y,n,n,n,y,y,democrat
23,y,y,y,n,n,n,y,y,y,n,n,n,n,n,y,y,democrat
25,y,n,y,n,n,n,y,y,y,y,n,n,n,n,y,y,democrat


In [7]:
df_chess.shape

(3196, 36)

In [8]:
# find binary variables in chess df
binary_vars_chess = [col for col in df_chess.columns if df_chess[col].nunique() == 2]
print(f"Binary variables in Chess dataset: {binary_vars_chess}")

Binary variables in Chess dataset: ['bkblk', 'bknwy', 'bkon8', 'bkona', 'bkspr', 'bkxbq', 'bkxcr', 'bkxwp', 'blxwp', 'bxqsq', 'cntxt', 'dsopp', 'dwipd', 'katri', 'mulch', 'qxmsq', 'r2ar8', 'reskd', 'reskr', 'rimmx', 'rkxwp', 'rxmsq', 'simpl', 'skach', 'skewr', 'skrxp', 'spcop', 'stlmt', 'thrsk', 'wkcti', 'wkna8', 'wknck', 'wkovl', 'wkpos', 'wtoeg']


In [9]:
sdp_voting = naive_bayes_sdp(
    model=voting_model,
    D=target_voting,
    d_value='democrat',
    evidence={},
    threshold=0.5,
)

In [29]:
sdp_voting_fast = fast_broadcast_sdp(
    model=voting_model,
    D=target_voting,
    d_value='republican',
    evidence={},
    threshold=0.5,
    partitions= get_partitions(voting_model, voting_model.nodes(), target_voting, {})
)

In [30]:
sdp_voting

0.5343369815465464

In [12]:
alarm_model = get_example_model('alarm')
child_model = get_example_model('child')
#asia_model = get_example_model('asia')
insurance_model = get_example_model('insurance')
hailfinder_model = get_example_model('hailfinder')
hepar_model = get_example_model('hepar2')
barley_model = get_example_model('barley')
win95pts_model = get_example_model('win95pts')
#mildew_model = get_example_model('mildew')
#water_model = get_example_model('water')
mildew_model = None
water_model = None

In [13]:
models = {
    'medium': [alarm_model, child_model, insurance_model, barley_model, mildew_model, water_model],
    'large': [hailfinder_model, hepar_model, win95pts_model]
}

In [14]:
child_model.name = 'child'
insurance_model.name = 'insurance'
alarm_model.name = 'alarm'
hepar_model.name = 'hepar'
hailfinder_model.name = 'hailfinder'
win95pts_model.name = 'win95pts'
barley_model.name = 'barley'
voting_model.name = 'voting'
chess_model.name = 'chess'
#mildew_model.name = 'mildew'
#water_model.name = 'water'

In [15]:
def rename_state_mapping(model):
    state_mapping = {'yes': 1, 'no': 0, 'True': 1, 'False': 0, 'TRUE':0, 'FALSE':1, 'present': 1, 'absent': 0, 'Present': 1, 'Absent': 0}
    for cpd in model.get_cpds():
        new_state_names = {}
        new_name_to_no = {}
        
        for var, states in cpd.state_names.items():
            # Map the old states ('yes'/'no') to the new states (1/0)
            # If a state isn't in our mapping, it stays the same
            new_states = [state_mapping.get(state, state) for state in states]
            
            # Update the dictionaries
            new_state_names[var] = new_states
            new_name_to_no[var] = {state: idx for idx, state in enumerate(new_states)}
            
        # Assign the updated dictionaries back to the CPD
        cpd.state_names = new_state_names
        cpd.name_to_no = new_name_to_no
    
    return model

In [16]:
#for size in ['medium', 'large']:
#    for model in models[size]:
#        model = rename_state_mapping(model)

# Monte Carlo Single Tests

In [50]:
import random
from pgmpy.models import BayesianNetwork
from pgmpy.inference import VariableElimination

def find_exact_experimental_patients(bn, target_node, target_value, decision_threshold, evidence_vars, buckets=[0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0], tolerance=0.05, batch_size=8000, max_batches=10):
    """
    Brute-force searches for patients by generating massive random batches.
    Evaluates each random reality and assigns it to a bucket if the exact SDP matches.
    Returns a dictionary mapping buckets to a tuple: (patient_evidence_dict, exact_sdp)
    """
    all_nodes = list(bn.nodes())
    hidden_vars = [n for n in all_nodes if n not in evidence_vars and n != target_node]
    
    print(f"\nHunting for patients... (Locking {len(evidence_vars)} variables as evidence)")
    
    # ==========================================
    # OPTIMIZATION: Pruned Sub-model for Base Decisions
    # ==========================================
    relevant_nodes = list(evidence_vars) + [target_node]
    ancestral_structure = bn.get_ancestral_graph(relevant_nodes)
    
    sub_model = BayesianNetwork(ancestral_structure.edges())
    sub_model.add_nodes_from(ancestral_structure.nodes())
    for node in sub_model.nodes():
        sub_model.add_cpds(bn.get_cpds(node))
        
    base_inference = VariableElimination(sub_model)
    # ==========================================
    
    unfilled_buckets = {b: None for b in buckets}
    batch_count = 0
    
    while any(v is None for v in unfilled_buckets.values()) and batch_count < max_batches:
        batch_count += 1
        print(f"Generating batch {batch_count}/{max_batches} of {batch_size} random realities...")
        
        for _ in range(batch_size):
            # 1. Generate random patient
            temp_patient = {}
            for var in evidence_vars:
                states = sub_model.get_cpds(var).state_names[var]
                temp_patient[var] = random.choice(states)
            
            # 2. Check base decision (Must be >= threshold)
            try:
                base_dist = base_inference.query(variables=[target_node], evidence=temp_patient, show_progress=False)
                if base_dist.get_value(**{target_node: target_value}) < decision_threshold:
                    continue # Reject and generate a new random patient
            except (ValueError, MemoryError):
                print(f"    [!] EXACT INFERENCE IMPOSSIBLE: Sub-network exceeded hardware limits.")
                return unfilled_buckets # Bail out safely
                
            # 3. Calculate Exact SDP
            partitions = get_partitions(bn, hidden_vars, target_node, temp_patient)
            #print(partitions)
            try:
                exact_sdp = fast_broadcast_sdp(bn, target_node, target_value, temp_patient, decision_threshold, partitions)
                #print(exact_sdp)
            except (ValueError, MemoryError):
                print(f"    [!] EXACT SDP IMPOSSIBLE: Tensor exploded during calculation.")
                return unfilled_buckets # Bail out safely
                
            # 4. Check if it fits into any empty bucket!
            empty_targets = [b for b, v in unfilled_buckets.items() if v is None]
            for b in empty_targets:
                if abs(exact_sdp - b) <= tolerance:
                    # Save the result as a tuple for your loop unpacking!
                    unfilled_buckets[b] = (temp_patient.copy(), exact_sdp)
                    print(f"--> Filled bucket {b} with Exact SDP: {exact_sdp:.4f}")
                    break # Only fill one bucket per patient
                    
            # Break the batch loop early if we filled everything
            if not any(v is None for v in unfilled_buckets.values()):
                break
                
    if any(v is None for v in unfilled_buckets.values()):
        missing = [b for b, v in unfilled_buckets.items() if v is None]
        print(f"Finished searching. Could not find patients for buckets: {missing}")
    else:
        print("All buckets filled successfully!")
        
    return unfilled_buckets

In [18]:
def run_variance_experiment(bn, target, target_value, threshold, experimental_patients, trials=20, mc_samples=11000):
    results = []
    
    for bucket, data in experimental_patients.items():
        true_sdp = data['true_sdp']
        evidence = data['evidence']
        print(f"\nRunning {trials} MC trials for patient with Exact SDP: {true_sdp:.4f}...")
        
        for trial in range(trials):
            est_sdp = mcmc_sdp_estimation(bn, target, target_value, evidence, threshold, n_samples=mc_samples, burn_in=1000, thinning=10)
            error = est_sdp - true_sdp
            
            results.append({
                'True_SDP': true_sdp,
                'Estimated_SDP': est_sdp,
                'Error': error,
                'Bucket': bucket
            })
            
    return pd.DataFrame(results)

In [19]:
def generate_patient_evidence(model, target, latent_size=10):
    all_variables = model.nodes()

    possible_evidence_vars = [v for v in all_variables if v != target]
    num_observed = len(all_variables) - latent_size - 1
    observed_vars = random.sample(possible_evidence_vars, num_observed)
    
    evidence = {}
    for var in observed_vars:
        # Fetch valid states for this specific variable from the BN
        valid_states = model.get_cpds(var).state_names[var]
        evidence[var] = random.choice(valid_states)
        
    return evidence

# Run Experiment

In [40]:
def get_target(model):
    targets = {
        'child': 'Sick',
        'alarm': 'HYPOVOLEMIA',
        'barley': 'pesticid',
        'insurance': 'Theft',
        'mildew': None, # no binary variables
        'water': None, # no binary variables
        'hailfinder': 'ScenRelAMCIN',
        'hepar': 'hepatomegaly',
        'win95pts': 'PrtMem',
        'voting': 'Class',
        'chess': 'skach'
    }
    # define the target manually when avaiable (from the respective paper) or randomly
    # conferir se vão ser esses mesmos!!

    return targets[model.name]

def get_h_ratio(model):
    ratios = {
        'child': 0.5,
        'alarm': 0.20,
        'hepar': 0.20,
        'barley': 0.20,
        'mildew': 0.20,
        'water': 0.20,
        'hailfinder': 0.20,
        'win95pts': 0.20,
        'insurance': 0.20,
        'voting': 0.5,
        'chess': 0.86
    }
    return ratios[model.name]
    



In [41]:
len(chess_model.nodes())

36

In [42]:
models_to_run = [voting_model, chess_model, child_model, alarm_model, barley_model, insurance_model, hailfinder_model, hepar_model, win95pts_model]

In [43]:
len(models_to_run)

9

In [44]:
all_targets_are_binary = True
for bn in models_to_run:
    #print(f"\n=== BN: {bn.name} ===")
    target = get_target(bn)
    if target is None:
        #print(f"--> No binary target defined for {bn.name}, skipping.")
        continue
    target_states = bn.get_cpds(target).state_names[target]
    if len(target_states) != 2:
        #print(f"--> Target '{target}' in {bn.name} is not binary (States: {target_states}), skipping.")
        all_targets_are_binary = False
        continue
    #print(f"Available states for target '{target}': {target_states}")
    target_value = target_states[1] if len(target_states) > 1 else target_states[0]
    #print(f"Target Node: {target}, Target Value: {target_value}")

print(f"\nAll targets are binary: {all_targets_are_binary}")


All targets are binary: True


In [45]:
for bn in models_to_run:
    print(f"\n=== BN: {bn.name} ===")
    all_nodes = list(bn.nodes())
    
    target = get_target(bn)
    if target is None:
        print(f"--> No binary target defined for {bn.name}, skipping.")
        continue
    target_states = bn.get_cpds(target).state_names[target]
    target_value = target_states[1] if len(target_states) > 1 else target_states[0]
    print(f"Target Node: {target}, Target Value: {target_value}")
    
    available_nodes = [n for n in all_nodes if n != target]
    print(f"H ratio: {get_h_ratio(bn)}")
    n_hidden = max(1, int(len(available_nodes) * get_h_ratio(bn)))
    print(f"using {n_hidden} H variables")


=== BN: voting ===
Target Node: Class, Target Value: republican
H ratio: 0.5
using 8 H variables

=== BN: chess ===
Target Node: skach, Target Value: t
H ratio: 0.86
using 30 H variables

=== BN: child ===
Target Node: Sick, Target Value: no
H ratio: 0.5
using 9 H variables

=== BN: alarm ===
Target Node: HYPOVOLEMIA, Target Value: FALSE
H ratio: 0.2
using 7 H variables

=== BN: barley ===
Target Node: pesticid, Target Value: x_2
H ratio: 0.2
using 9 H variables

=== BN: insurance ===
Target Node: Theft, Target Value: False
H ratio: 0.2
using 5 H variables

=== BN: hailfinder ===
Target Node: ScenRelAMCIN, Target Value: CThruK
H ratio: 0.2
using 11 H variables

=== BN: hepar ===
Target Node: hepatomegaly, Target Value: absent
H ratio: 0.2
using 13 H variables

=== BN: win95pts ===
Target Node: PrtMem, Target Value: Less_than_2Mb
H ratio: 0.2
using 15 H variables


In [60]:
import time
from xml.parsers.expat import model
def run_targeted_sdp_experiment(output_csv="targeted_sdp_benchmark.csv"):
    
    results = []
    raw_results = []
    #H_RATIO = 0.20
    DECISION_THRESHOLD = 0.5
    TARGET_BUCKETS = [0.40, 0.50, 0.6, 0.70, 0.8, 0.9, 1.0]
    MCMC_TRIALS = 20 
    
    for bn in models_to_run:
        n_nodes = bn.number_of_nodes()
        print(f"\n========================================")
        print(f"Processing BN: {bn.name}")
        
        all_nodes = list(bn.nodes())
        
        target = get_target(bn)
        if target is None:
            print(f"--> No binary target defined for {bn.name}, skipping.")
            continue
        target_states = bn.get_cpds(target).state_names[target]
        target_value = target_states[1] if len(target_states) > 1 else target_states[0]
        print(f"Target Node: {target}, Target Value: {target_value}")
        
        available_nodes = [n for n in all_nodes if n != target]
        print(f"H ratio: {get_h_ratio(bn)}")
        n_hidden = max(1, int(len(available_nodes) * get_h_ratio(bn)))
        print(f"using {n_hidden} H variables")
        hidden_vars = random.sample(available_nodes, n_hidden)
        evidence_vars = [n for n in available_nodes if n not in hidden_vars]
        
        # Run the Harvester 
        #harvested_data = harvest_patients_for_all_buckets(
        #    bn, target, target_value, DECISION_THRESHOLD, evidence_vars, TARGET_BUCKETS
        #)
        if bn.name == 'ignore':
            # use only 0.5 bucket
            harvested_data = find_exact_experimental_patients(bn, target, target_value, DECISION_THRESHOLD,
                                                          evidence_vars, buckets=[0.5])
        else:
            harvested_data = find_exact_experimental_patients(bn, target, target_value, DECISION_THRESHOLD,
                                                          evidence_vars, buckets=TARGET_BUCKETS)
        
        # Now process whatever it managed to find
        for target_sdp, result in harvested_data.items():
            if result is None:
                continue # We didn't find a patient for this specific bucket in this network
                
            patient, exact_sdp = result
            print(f"\n  -> Benchmarking found patient for bucket {target_sdp} (Exact: {exact_sdp:.4f})")
            
            # ========================================================
            # RACE TIMING 1: EXACT SDP
            # ========================================================
            partitions = get_partitions(bn, hidden_vars, target, patient)
            exact_time = np.nan
            
            try:
                start_time = time.time()
                # Re-run the exact calculation once just to time it cleanly
                exact_sdp_benchmark = fast_broadcast_sdp(bn, target, target_value, patient, DECISION_THRESHOLD, partitions)
                exact_time = time.time() - start_time
                print(f"       -> Exact Time: {exact_time:.4f} seconds")
            except (ValueError, MemoryError):
                print(f"       -> Exact Time: [FAILED DUE TO MEMORY/EINSUM LIMIT]")
            
            # ========================================================
            # RACE TIMING 2: MCMC SDP
            # ========================================================
            mcmc_estimates = []
            mcmc_times = []
            
            for trial in range(MCMC_TRIALS):
                start_time = time.time()
                est_sdp = fast_mcmc_sdp_estimation(
                    bn, target, target_value, patient, DECISION_THRESHOLD,
                    n_samples=7000, burn_in=500, thinning=5
                )
                mcmc_times.append(time.time() - start_time)
                mcmc_estimates.append(est_sdp)
                raw_results.append({
                    'Network': bn.name,
                    'Target_Bucket': target_sdp,
                    'Exact_SDP': exact_sdp,
                    'MCMC_Estimate': est_sdp,
                    'Exact_Time_sec': exact_time, 
                    'MCMC_Time_sec': mcmc_times[-1]
                })
                
            mcmc_mean = np.mean(mcmc_estimates)
            mcmc_variance = np.var(mcmc_estimates)
            mcmc_avg_time = np.mean(mcmc_times)
            
            print(f"       -> MCMC Mean Estimate: {mcmc_mean:.4f}")
            print(f"       -> MCMC Avg Time: {mcmc_avg_time:.4f} seconds")
            
            absolute_error = abs(exact_sdp - mcmc_mean)
            
            # Record everything to the dataset
            results.append({
                'Network': bn.name,
                'N_Nodes': n_nodes,
                'Target_Bucket': target_sdp,
                'Target_Node': target,
                'Target_Value': target_value,
                'Exact_SDP': exact_sdp,
                'Exact_Time_sec': exact_time,
                'MCMC_Mean_SDP': mcmc_mean,
                'MCMC_Variance': mcmc_variance,
                'MCMC_Avg_Time_sec': mcmc_avg_time,
                'Absolute_Error': absolute_error
            })
            
            # Save progressively
            pd.DataFrame(results).to_csv(output_csv, index=False)
            pd.DataFrame(raw_results).to_csv("raw_" + output_csv, index=False)

    print(f"\nExperiment Complete! Results saved to {output_csv}")
    return pd.DataFrame(results)

In [ ]:
run_targeted_sdp_experiment()


Processing BN: voting
Target Node: Class, Target Value: republican
H ratio: 0.5
using 8 H variables

Hunting for patients... (Locking 8 variables as evidence)
Generating batch 1/10 of 8000 random realities...
--> Filled bucket 1.0 with Exact SDP: 0.9757
--> Filled bucket 0.6 with Exact SDP: 0.6446
--> Filled bucket 0.9 with Exact SDP: 0.8641
--> Filled bucket 0.8 with Exact SDP: 0.7901
--> Filled bucket 0.5 with Exact SDP: 0.5341
--> Filled bucket 0.7 with Exact SDP: 0.6527
Generating batch 2/10 of 8000 random realities...
Generating batch 3/10 of 8000 random realities...
Generating batch 4/10 of 8000 random realities...
Generating batch 5/10 of 8000 random realities...


In [102]:
df = pd.read_csv('targeted_sdp_benchmark.csv')
print("TOP 10 HIGHEST-ERROR CASES")
print("=" * 60)
cols = ["Network", "N_Nodes",
        "Target_Bucket", "Exact_SDP", "MCMC_Mean_SDP",
        "Absolute_Error", "MCMC_Variance"]
print(df.nlargest(10, "Absolute_Error")[cols].to_string(index=False))

TOP 10 HIGHEST-ERROR CASES
Network  N_Nodes  Target_Bucket  Exact_SDP  MCMC_Mean_SDP  Absolute_Error  MCMC_Variance
  hepar       70            0.5   0.518348       0.509171        0.009176       0.000077
  child       20            0.6   0.629798       0.637857        0.008059       0.000741
  alarm       37            0.6   0.560922       0.567457        0.006535       0.000056
  child       20            0.8   0.782735       0.788429        0.005693       0.000192
  child       20            0.7   0.669727       0.664543        0.005184       0.000343
  hepar       70            0.6   0.555990       0.560971        0.004981       0.000077
  child       20            0.5   0.502504       0.506143        0.003639       0.000439
  hepar       70            0.9   0.855005       0.858114        0.003110       0.000003
  child       20            0.9   0.909760       0.906714        0.003046       0.000015
  alarm       37            0.9   0.881610       0.878686        0.002924       0.0

In [103]:
df.shape

(17, 11)